# JusticeEngine-01: TRL GRPO Training Loop

This notebook trains an LLM using **GRPO** via `TRL` + `Unsloth` for 4-bit quantization.

**Setup:** Runtime > Change runtime type > **T4 GPU**

Full notebook with step-by-step cells: [`training/training_colab.ipynb`](https://colab.research.google.com/github/rishitaramola/judicial-reasoning-env/blob/main/training/training_colab.ipynb)

In [ ]:
# 1. Install Dependencies (Unsloth must be installed FIRST)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install datasets wandb gymnasium pydantic python-dotenv

In [ ]:
# 2. Verify GPU is available (REQUIRED for Unsloth)
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

### Setup Credentials
Add your `HF_TOKEN` to Colab Secrets (key icon in left sidebar).

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF_TOKEN loaded from Colab Secrets")
except Exception:
    print("HF_TOKEN not found. Model will be saved locally only.")

In [ ]:
# 3. Clone the Repository
%cd /content
!rm -rf judicial-reasoning-env
!git clone https://github.com/rishitaramola/judicial-reasoning-env.git
%cd judicial-reasoning-env

In [ ]:
# 4. Quick Sanity Check: Verify the RL environment and reward functions work
!python -c "from environment import JudicialEnv; env = JudicialEnv('contract','easy'); obs,_=env.reset(); print(f'ENV OK: {obs.case_id}')"
!python -c "from graders.programmatic_grader import ProgrammaticGrader; g=ProgrammaticGrader(); print('GRADER OK:', g.grade_all())"

### Run the Training Script
The `train.py` script runs 3-phase curriculum training (250 steps total, ~30-45 min on T4).

In [ ]:
# 5. Execute the Full GRPO Training Loop (250 steps, ~30-45 min on T4)
!python train.py